In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path.cwd().parent / "data"
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
COLOR = {"Fresh": "#2a78d6", "Style": "#eb6834", "Tech": "#1baf7a"}
GROUPS = ["Fresh · chilled", "Fresh · ambient", "Style", "Tech"]
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titlelocation": "left", "axes.titlepad": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "font.size": 10,
})

In [ ]:
orders = pd.concat([pd.read_csv(DATA / "Training Data/deliveries_train.csv"),
                    pd.read_csv(DATA / "Test Data/task1_test_inputs.csv")], ignore_index=True)
is_def = orders.dispatch_status == "deferred"
print(f"{len(orders):,} orders, {is_def.sum():,} deferred")

In [ ]:
g = pd.Series(np.where(orders.brand == "Fresh", "Fresh · " + orders.temp_requirement, orders.brand), index=orders.index)
total = g.groupby(g).size().reindex(GROUPS)
deferred = g[is_def].pipe(lambda s: s.groupby(s).size()).reindex(GROUPS, fill_value=0)
rate = deferred / total

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
colors = [COLOR[grp.split(" · ")[0]] for grp in GROUPS]
x = np.arange(len(GROUPS))
bars = ax1.bar(x, deferred.values, color=colors, width=0.6)
ax1.bar_label(bars, labels=[f"{n:,}" for n in deferred.values], padding=3, fontsize=9.5, color=INK_2)
ax1.set_title("Deferred orders")
ax1.set_ylabel("Orders")
ax1.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
bars = ax2.bar(x, rate.values * 100, color=colors, width=0.6)
ax2.bar_label(bars, labels=[f"{r:.2%}\n({d:,} of {t:,})" for r, d, t in zip(rate, deferred, total)],
              padding=3, fontsize=9, color=INK_2)
ax2.set_title("Share of the brand's orders deferred")
ax2.set_ylabel("% of orders")
ax2.yaxis.set_major_formatter(lambda v, _: f"{v:g}%")
for ax in (ax1, ax2):
    ax.set_xticks(x, GROUPS)
    ax.grid(axis="x", visible=False)
    ax.margins(y=0.2)
fig.suptitle(f"Deferred orders by brand ({deferred.sum():,} of {total.sum():,} orders)",
             x=0.07, ha="left", fontsize=14, color=INK, y=1.03)
plt.show()
pd.DataFrame({"orders": total, "deferred": deferred, "rate": rate.round(4)})

In [ ]:
dow = pd.to_datetime(orders.order_date).dt.dayofweek
total = orders.groupby(dow).size().reindex(range(7), fill_value=0)
deferred = orders[is_def].groupby(dow[is_def]).size().reindex(range(7), fill_value=0)
days = orders.groupby(dow).order_date.nunique().reindex(range(7), fill_value=0)
rate = (deferred / total.replace(0, np.nan)).fillna(0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
x = np.arange(7)
bars = ax1.bar(x, deferred.values, color=COLOR["Fresh"], width=0.62)
ax1.bar_label(bars, labels=[f"{n:,}" for n in deferred.values], padding=3, fontsize=9.5, color=INK_2)
ax1.set_title("Deferred orders")
ax1.set_ylabel("Orders")
ax1.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
bars = ax2.bar(x, rate.values * 100, color=COLOR["Fresh"], width=0.62)
ax2.bar_label(bars, labels=[f"{r:.1%}" if t else "no orders" for r, t in zip(rate, total)],
              padding=3, fontsize=9.5, color=INK_2)
ax2.set_title("Share of that weekday's orders deferred")
ax2.set_ylabel("% of orders")
ax2.yaxis.set_major_formatter(lambda v, _: f"{v:g}%")
for ax in (ax1, ax2):
    ax.set_xticks(x, DAYS)
    ax.grid(axis="x", visible=False)
    ax.margins(y=0.15)
fig.suptitle(f"Deferred orders by day of week ({deferred.sum():,} orders)", x=0.07, ha="left", fontsize=14, color=INK, y=1.03)
plt.show()
pd.DataFrame({"days": days, "orders": total, "deferred": deferred,
              "per_day": (deferred / days.replace(0, np.nan)).round(2), "rate": rate.round(4)}).set_axis(DAYS)